# 🔎 Georgia RAG — pipeline step by step

This notebook lets you walk through the whole pipeline by hand and inspect what comes out at each stage:

1. Check `.env` and settings
2. (optional) Fetch chat history
3. Raw messages
4. Spam cleaning (money / drugs / ads / pets; questions are kept)
5. Knowledge distillation (threads → Q&A)
6. Validate the distillation (LLM judge)
7. Fix — apply the judge's verdicts
8. Итоговые знания (this preview batch)
9. Все итоговые знания (с диска)
10. Инкрементальное обновление (update_knowledge)
11. Indexing
12. Retrieval
13. RAG answer
14. Debug: which prompt is actually sent to GPT

> Run: `uv run jupyter lab` from the project root, then open `notebooks/explore.ipynb`.

> Note: query strings stay in Russian on purpose — they must match the Russian content of the chats.


In [92]:
# So that `import config` / `src.*` work from the notebooks/ folder, move to the project root
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

# Autoreload: edits in src/*.py are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

Working directory: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Check `.env` and settings

Make sure the keys were picked up (values are masked).

In [93]:
import importlib, config
importlib.reload(config)

def mask(v):
    return (v[:4] + "…" + str(len(v)) + " chars") if v else "❌ not set"

print("OPENAI_API_KEY  :", mask(config.OPENAI_API_KEY))
print("BOT_TOKEN       :", mask(config.BOT_TOKEN))
print("TELEGRAM_API_ID :", config.TELEGRAM_API_ID or "❌ not set")
print("TELEGRAM_PHONE  :", config.TELEGRAM_PHONE or "❌ not set")
print()
print("USE_AZURE_OPENAI:", config.USE_AZURE_OPENAI)
if config.USE_AZURE_OPENAI:
    print("  AZURE_OPENAI_ENDPOINT:", config.AZURE_OPENAI_ENDPOINT or "❌ not set")
    print("  AZURE_OPENAI_API_KEY :", mask(config.AZURE_OPENAI_API_KEY))
print()
print("Chats:", [c["username"] for c in config.CHATS])
print("Pipeline models:", config.SPLIT_MODEL, "|", config.JUDGE_MODEL, "|", config.FIX_MODEL, "|", config.REVERIFY_MODEL, "|", config.GENERATION_MODEL)
print("Thread caps: gap =", config.CHUNK_MAX_GAP_MINUTES, "min, max burst size =", config.THREAD_MAX_BURST_SIZE)


OPENAI_API_KEY  : sk-p…164 chars
BOT_TOKEN       : 8624…46 chars
TELEGRAM_API_ID : 38918746
TELEGRAM_PHONE  : +995XXXXXXXXX

Chats: ['helpgeorgia', 'ipgeorgiachat']
Models: text-embedding-3-small | gpt-4o-mini
Chunking: max gap = 10 min, max size = 1500 chars


## 2. (optional) Fetch chat history

If you already ran `uv run python -m src.ingest` in the terminal — skip this step.

The first run will ask for the confirmation code from Telegram (entered right in the notebook). After authorization a session file is created, so the code won't be needed again.

In [94]:
# Uncomment to fetch history directly from the notebook:
#
# from telethon import TelegramClient
# from src.ingest import ingest_chat
#
# client = TelegramClient("georgia_ingest", int(config.TELEGRAM_API_ID), config.TELEGRAM_API_HASH)
# await client.start(phone=config.TELEGRAM_PHONE or None)
# for chat in config.CHATS:
#     await ingest_chat(client, chat)
# await client.disconnect()

## 3. Raw messages

Look at what was fetched: how many messages and what they look like.

In [95]:
from src.preprocess import _load_raw

username = "nogotochki"  # <- поменяй, чтобы переключить чат (должно совпадать с разделом 5)
raw_path = config.RAW_DIR / f"{username}.jsonl"
print("File:", raw_path, "| exists:", raw_path.exists())

if raw_path.exists():
    msgs = _load_raw(raw_path)
    print("Total messages:", len(msgs))
    print("\nLast 5:")
    for m in msgs[-5:]:
        sender = m.get("sender") or "Anonymous"
        print(f"  [{m['date'][:16]}] {sender}: {m['text'][:90]}")
else:
    print("Fetch the history first (step 2 or `uv run python -m src.ingest`).")

File: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag/data/raw/helpgeorgia.jsonl | exists: True
Total messages: 2724

Last 5:
  [2026-09-05T10:45] Stanislav: Тбилиси, переезжаем в пустую квартиру, где почти ничего нет. Может, кто-нибудь отдает нену
  [2026-09-06T10:30] Mariam: Avosend тоже работает
  [2026-09-07T09:43] Mariam: Здравствуйте всем. Подскажите пожалуйста, есть ли в Батуми магазины для беременных? Это дл
  [2026-09-10T09:57] Настасья: Здравствуйте! 👋

Подскажите, пожалуйста, кто-нибудь получал справку/подтверждение о том, ч
  [2026-09-10T10:49] Рома: здоавствуйте, подскажите где найти рускоговорящего юриста в грузии?


## 4. Spam cleaning

Before distillation we drop spam: "quick money" schemes, veiled drug ads from darkstores, self-promo / advertising, and pet-rehoming ads. **Questions and advice are always kept** (a guard skips matches that look like "подскажите / где можно / кто знает ..."). Patterns live in `src/spam.py` (`SPAM_PATTERNS`) — eyeball the removed examples below and tune them. This cell sets `msgs` to the cleaned list for everything that follows.

> In the pipeline this runs automatically (toggle with `config.FILTER_SPAM`).


In [96]:
from src.spam import filter_spam
from collections import Counter

clean, removed = filter_spam(msgs)
print(f"{len(msgs)} messages -> {len(clean)} clean, {len(removed)} spam removed")
print("by category:", dict(Counter(r["spam"] for r in removed)))

# A few examples of what got removed — eyeball these for false positives:
for r in removed:
    print(f"  [{r['spam']}] {r['text'][:100]}")

# Use the cleaned messages for everything below:
msgs = clean

2724 messages -> 2618 clean, 106 spam removed
by category: {'pets': 86, 'ads': 15, 'drugs': 5}
  [pets] ✨ Ищем заботливых хозяев для нашей милой кошечки Киры! ✨

Кира – настоящая пушистая радость, обожает
  [pets] Здравствуйте. Котята ищут дом. Уличная кошка родила в подъезде. Котята обработаны, знают лоток, едят
  [pets] Пишите @ yuliia_animals чтобы забрать малышку к себе 😍

Бесплатно в Добрые Руки ✨❤️
  [pets] Меня зовут Буся! 😌 Мне полгода. В мае меня нашли в подвале с травмированной лапкой: от ампутации я б
  [pets] Самый нежный в мире котенок Люда ищет дом ❤️

Она ласковая, мурчит и ложится на руки даже к незнаком
  [pets] Очаровательные котята, метисы британцев ищут любящие семьи ❤️

Их нашли на мусорке сразу всех вместе
  [pets] 🐾 Щенячий патруль в поисках заботливых хозяев! 🐾

Внимание, внимание! Наш отряд хвостатых непосед го
  [pets] Эриус — ваше лекарство от аллергии на осеннюю хандру 😊

Настала осень, пришла пора искать себе тепло
  [pets] Роскошная кошка Туся, победившая 

## 5. Knowledge distillation

Group messages into **threads** (reply links + time-neighbours) and let the LLM distill each thread into reusable **question → answer** knowledge, linked to the source message. Threads with no useful knowledge are skipped.

⚠️ Spends OpenAI tokens (`config.SPLIT_MODEL` + `config.EXTRACT_MODEL`, two calls per thread). Start with a small `limit` and eyeball the result before scaling up. Prompts live in `src/knowledge.py`; thread building in `src/threads.py`.


In [102]:
from src.threads import build_threads

threads = build_threads(msgs)   # msgs = cleaned messages from step 3b
sizes = sorted((len(t) for t in threads), reverse=True)
multi = [s for s in sizes if s > 1]
print(f"{len(msgs)} messages -> {len(threads)} threads ({len(multi)} multi-message)")
print("top thread sizes:", sizes[:10])

# Look at one multi-message thread in full:
for t in threads:
    if len(t) >= 5:
        print("\n--- thread root", t[0]["link"], f"({len(t)} msgs) ---")
        for m in t:
            print(f"  {(m.get('sender') or '?')[:12]:12} | {m['text'][:60]}")
        break

2618 messages -> 1796 threads (349 multi-message)
top thread sizes: [13, 12, 11, 11, 11, 10, 10, 9, 9, 9]

--- thread root https://t.me/helpgeorgia/305564 (6 msgs) ---
  Omni         | Добрый вечер! Подскажите по заказу с Aliexpress - как удобне
  Tati         | С Китая в основном заказы с Таобао, а не с АлиЭкспресс. АлиЭ
  Omni         | То, что мне нужно не нашёл. На Temu тоже проверял. Нужно име
  Tati         | Там можно напрямую, на почту приходит, через неопределенное 
  Omni         | 3 месяца? )  ладно, понял, спасибо всё равно.
  Leila        | Тбилиси!Срочно нужна помощь по пристройству,котенок 2,5 меся


In [103]:
# Preview WHICH threads "first 20" actually means — before spending any tokens.
# select_threads() is the exact same selection distill_chat() uses internally
# (ascending root_msg_id, after dropping threads whose last message is older
# than config.INGEST_SINCE — see src/knowledge.py).
from src.knowledge import select_threads

chat = next(c for c in config.CHATS if c["username"] == "nogotochki")  # <- поменяй username, чтобы переключить чат
preview_threads = select_threads(
    chat["username"],
    limit=20,
    min_thread_size=2,
)
print(f"{len(preview_threads)} threads selected\n")
for t in preview_threads:
    root = t[0]
    print(f"root={root['msg_id']}  {root['date'][:10]}  ({len(t)} msg)  {root['link']}")


[knowledge] helpgeorgia: skipped 148 threads older than 2025-03-01
20 threads selected

root=308018  2025-02-27  (6 msg)  https://t.me/helpgeorgia/308018
root=308040  2025-03-05  (2 msg)  https://t.me/helpgeorgia/308040
root=308045  2025-03-06  (2 msg)  https://t.me/helpgeorgia/308045
root=308066  2025-03-10  (8 msg)  https://t.me/helpgeorgia/308066
root=308102  2025-03-13  (3 msg)  https://t.me/helpgeorgia/308102
root=308108  2025-03-14  (2 msg)  https://t.me/helpgeorgia/308108
root=308111  2025-03-14  (5 msg)  https://t.me/helpgeorgia/308111
root=308115  2025-03-15  (3 msg)  https://t.me/helpgeorgia/308115
root=308164  2025-03-19  (2 msg)  https://t.me/helpgeorgia/308164
root=308171  2025-03-20  (4 msg)  https://t.me/helpgeorgia/308171
root=308206  2025-03-22  (5 msg)  https://t.me/helpgeorgia/308206
root=308210  2025-03-23  (4 msg)  https://t.me/helpgeorgia/308210
root=308267  2025-03-30  (2 msg)  https://t.me/helpgeorgia/308267
root=308284  2025-04-01  (5 msg)  https://t.me/helpgeo

In [165]:
# ⚠️ Spends OpenAI tokens. Distill the first N multi-message threads to eyeball quality.
from src.knowledge import distill_chat

knowledge = distill_chat(
    chat["username"],
    limit=20,            # only the first 20 threads (cheap prototype)
    min_thread_size=2,   # discussions only; set 1 to also distill standalone facts
    write=False,         # don't write the jsonl yet, just inspect
)
print(f"\nGot {len(knowledge)} knowledge items from up to 20 threads")

[knowledge] helpgeorgia: skipped 148 threads older than 2025-03-01
[knowledge] helpgeorgia: 20/20 threads -> 13 items

Got 13 knowledge items from up to 20 threads


In [166]:
# View the distilled knowledge — eyeball quality / hallucinations / provenance
# root_link = начало ВСЕГО треда (может быть до 50 сообщений, часто не по теме);
# source_link = конкретное сообщение, откуда реально взят этот вопрос/ответ.
for k in knowledge[:15]:
    print("Q:", k["question"])
    print("A:", k["answer"])
    same = k["source_link"] == k["root_link"]
    print("  root:  ", k["root_link"])
    print("  source:", k["source_link"], "(= root)" if same else "(другое сообщение!)")
    print("-" * 70)


Q: Где в Тбилиси делается справка о несудимости?
A: Справка о несудимости в Тбилиси делается в соответствующих государственных учреждениях, информацию можно найти по ссылке.
  ↳ https://t.me/helpgeorgia/308018
----------------------------------------------------------------------
Q: Как записаться в сферу интересов РФ для оформления справки о несудимости в РФ?
A: Лучше всего позвонить в соответствующее учреждение, так как правила могут меняться.
  ↳ https://t.me/helpgeorgia/308018
----------------------------------------------------------------------
Q: Можно ли заказать справку о несудимости через госуслуги?
A: Да, справку о несудимости можно заказать через госуслуги, не нужно никуда ехать.
  ↳ https://t.me/helpgeorgia/308018
----------------------------------------------------------------------
Q: Сколько стоит обмотать чемодан плёнкой в аэропорту?
A: Около 30 лари.
  ↳ https://t.me/helpgeorgia/308045
----------------------------------------------------------------------
Q: Где в Тби

In [167]:
# "Было -> стало": для каждого выбранного треда — исходный текст И то, что LLM из него извлёк.
# Треды сопоставляются с юнитами знаний по root_msg_id.
from src.knowledge import _thread_text
from collections import defaultdict

by_root = defaultdict(list)
for k in knowledge:
    by_root[k["root_msg_id"]].append(k)

for t in preview_threads:
    root = t[0]
    units = by_root.get(root["msg_id"], [])
    print("=" * 70)
    print(f"ТРЕД {root['link']}  ({len(t)} msg)")
    print("--- было (исходный тред) ---")
    print(_thread_text(t)[:800])
    print(f"--- стало ({len(units)} извлечённых пар) ---")
    if not units:
        print("  (ничего не извлечено)")
    for u in units:
        print(f"  Q: {u['question']}")
        print(f"  A: {u['answer']}")
        print(f"  type: {u['type']}  city: {u.get('city')}")
        print(f"  ↓ source: {u['source_link']}")
    print()


ТРЕД https://t.me/helpgeorgia/308018  (6 msg)
--- было (исходный тред) ---
Alex: Ребят, а где в Тбилиси делается справка о несудимости? 🥸
Аноним: https://t.me/nlevshitstelegram/19384
Alex: А не подскажете, как записаться в сферу интересов РФ для оформления справки о несудимости именно в РФ? 🇷🇺
Аноним: Позвоните им) Правила могут меняться, так будет самое надёжное
Andrew: На госуслугах я заказывал эту справку. Никуда ехать не приходилось
Nadezhda P.: Апостиль был на справке ?
--- стало (3 извлечённых пар) ---
  Q: Где в Тбилиси делается справка о несудимости?
  A: Справка о несудимости в Тбилиси делается в соответствующих государственных учреждениях, информацию можно найти по ссылке.
  type: vote_based
  Q: Как записаться в сферу интересов РФ для оформления справки о несудимости в РФ?
  A: Лучше всего позвонить в соответствующее учреждение, так как правила могут меняться.
  type: vote_based
  Q: Можно ли заказать справку о несудимости через госуслуги?
  A: Да, справку о несудимости можн

## 6. Validate the distillation (LLM judge)

Run the same LLM-judge harness used for the full base (`src/eval_knowledge.py`) on just these 20 threads — cheap, and lets you see exactly what the judge outputs and whether it makes sense before trusting it on the full run.


In [168]:
# ⚠️ Spends OpenAI tokens (config.JUDGE_MODEL, ~1 call per thread — cheap for 20 threads).
from src.eval_knowledge import judge_units

threads_by_root = {t[0]["msg_id"]: t for t in preview_threads}
judged = judge_units(knowledge, threads_by_root)

# Три колонки в одном виде: ТРЕД -> дистиллировано -> вердикт судьи, сгруппировано по треду.
by_root_j = defaultdict(list)
for j in judged:
    by_root_j[j["root_msg_id"]].append(j)

for t in preview_threads:
    root = t[0]
    js = by_root_j.get(root["msg_id"], [])
    if not js:
        continue  # ничего не дистиллировано из этого треда — нечего судить
    print("=" * 70)
    print(f"ТРЕД {root['link']}  ({len(t)} msg)")
    print("--- было ---")
    print(_thread_text(t)[:500])
    for j in js:
        print(f"--- стало: [{j['type']}] city={j.get('city')} ---")
        print(f"  Q: {j['question']}")
        print(f"  A: {j['answer']}")
        print(f"--- судья: verdict={j['verdict']}  faithful={j['faithful']} atomic={j['atomic']} "
              f"useful={j['useful']} type_ok={j['type_ok']} (suggested: {j['type_suggested']}) "
              f"city_ok={j['city_ok']} (suggested: {j['city_suggested']})")
        if j["note"]:
            print(f"  note: {j['note']}")
    print()

# Сводка по этим 20 тредам:
from collections import Counter
print("verdict:", dict(Counter(j["verdict"] for j in judged)))


[eval:precision] 13/13
ТРЕД https://t.me/helpgeorgia/308018  (6 msg)
--- было ---
Alex: Ребят, а где в Тбилиси делается справка о несудимости? 🥸
Аноним: https://t.me/nlevshitstelegram/19384
Alex: А не подскажете, как записаться в сферу интересов РФ для оформления справки о несудимости именно в РФ? 🇷🇺
Аноним: Позвоните им) Правила могут меняться, так будет самое надёжное
Andrew: На госуслугах я заказывал эту справку. Никуда ехать не приходилось
Nadezhda P.: Апостиль был на справке ?
--- стало: [vote_based] ---
  Q: Где в Тбилиси делается справка о несудимости?
  A: Справка о несудимости в Тбилиси делается в соответствующих государственных учреждениях, информацию можно найти по ссылке.
--- судья: verdict=drop  faithful=False atomic=True useful=True type_ok=True (suggested: vote_based)
  note: Ответ не подтверждается тредом. В треде нет информации о том, где в Тбилиси делается справка о несудимости. Ссылка, предоставленная в ответе, не объяснена и не подтверждает информацию.
--- стало: [v

## 7. Fix — apply the judge's verdicts

Turns eval_knowledge's verdicts into an actual fixed dataset: adopts the suggested type, re-splits non-atomic pairs, and for faithful=false drops gets a second opinion from `config.REVERIFY_MODEL` before discarding — only drops when both judges agree; a disagreement means the pair is kept (and counted separately as `rescued` so you can see which keeps came from a disagreement).


In [ ]:
# ⚠️ Spends OpenAI tokens (one call per atomize + one per faithful=false drop).
from src.fix_knowledge import fix_batch

result = fix_batch(knowledge, judged, threads_by_root)

print(f"kept: {len(result['kept'])}  (rescued by 2nd judge: {len(result['rescued'])})")
print(f"fixed: {len(result['fixed'])}")
print(f"dropped: {len(result['dropped'])}")

print("\n=== FIXED (type/city corrected / re-atomized) ===")
for u in result['fixed']:
    print(f"  [{u['type']}] city={u.get('city')} Q: {u['question']}")
    print(f"        A: {u['answer'][:150]}")

print("\n=== RESCUED (2nd judge disagreed → kept) ===")
for u in result['rescued']:
    print(f"  Q: {u['question']}")
    print(f"     1st judge (drop): {u['judge_note']}")
    print(f"     2nd judge (keep): {u['reverify_note']}")

print("\n=== DROPPED (both judges agreed unfaithful, or useful=false) ===")
for u in result['dropped']:
    print(f"  Q: {u['question']}")
    print(f"     reason: {u.get('drop_reason', '')}")
    if u.get('reverify_note'):
        print(f"     2nd judge confirmed: {u['reverify_note']}")


## 8. Итоговые знания

`result['kept']` уже включает в себя `rescued` (спасённые вторым судьёй), поэтому финальный набор — это просто `kept + fixed`, без дублей.


In [ ]:
from src.fix_knowledge import save_fixed

final_knowledge = result['kept'] + result['fixed']
print(f"итог: {len(final_knowledge)} знаний из {len(knowledge)} исходных (distilled) / {len(preview_threads)} тредов\n")

for u in final_knowledge:
    print(f"[{u['type']}] city={u.get('city')} {u['question']}")
    print(f"  {u['answer']}")
    print(f"  ↳ {u['root_link']}")
    print()

save_fixed(chat['username'], result)


## 9. Все итоговые знания (с диска)

В отличие от раздела 8 (это только 20 превью-тредов в памяти) — здесь загружается весь файл `data/knowledge/<chat>.fixed.jsonl`, как он есть на диске после полного прогона (distill → eval → fix → save_fixed).


In [104]:
import json

fixed_path = config.KNOWLEDGE_DIR / f"{chat['username']}.fixed.jsonl"
with fixed_path.open(encoding='utf-8') as f:
    all_knowledge = [json.loads(line) for line in f if line.strip()]

print(f"{len(all_knowledge)} знаний в {fixed_path}\n")

for u in all_knowledge:
    print(f"[{u['type']}] city={u.get('city')}  {u['question']}")
    print(f"  {u['answer']}")
    print(f"  ↳ {u['root_link']}  ({u.get('date', '')[:10]})")
    print()


106 знаний в /Users/aleksandra/Documents/GitHub/telegram-georgia-rag/data/knowledge/helpgeorgia.fixed.jsonl

[date_based] city=None  Сколько стоит обмотать чемодан плёнкой в аэропорту?
  Около 30 лари.
  ↳ https://t.me/helpgeorgia/308045  (2025-03-06)

[vote_based] city=Тбилиси  Где в Тбилиси можно сдать использованные батарейки?
  Использованные батарейки принимают в East Point в главном здании на первом этаже. Также их можно сдать в кафе «Интроверт» во время свопов: там стоят боксы для батареек и вейпов, откуда их отправляют в «Парки ар минда». Свопы проходят в последние выходные каждого месяца.
  ↳ https://t.me/helpgeorgia/308066  (2025-03-10)

[vote_based] city=Батуми  Где в Батуми купить обычный двухколёсный самокат для взрослого?
  Можно посмотреть магазины с велосипедами, самокатами и скейтами на улице Пушкина, после перекрёстка улиц 26 Мая и Пушкина, если идти в сторону канатной дороги.
  ↳ https://t.me/helpgeorgia/308066  (2025-03-10)

[date_based] city=None  Сколько стоит гра

## 10. Инкрементальное обновление (update_knowledge)

Альтернатива полному прогону (разделы 5–9) — переобрабатывает ТОЛЬКО треды, активные с `processed_until - config.REPROCESS_OVERLAP_DAYS` (сейчас 3 дня), остальные треды не трогает. Первый запуск без файла состояния (`<chat>.state.json`) ведёт себя как полный прогон.

⚠️ Тратит токены (distill+eval+fix на отобранных тредах) и **перезаписывает** `data/knowledge/<chat>.jsonl` и `.fixed.jsonl` на диске (мерж с уже существующим содержимым). Если курсор уже забутстраплен и с последнего прогона не было новых сообщений в тредах — увидишь «nothing to (re)process».


In [105]:
from src.update_knowledge import _load_state, _state_path

print("current state:", _load_state(chat['username']))
print("state file:", _state_path(chat['username']))

# Раскомментируй, чтобы реально запустить (тратит токены, если есть что пересчитывать):
# from src.update_knowledge import update_knowledge
# update_knowledge(chat['username'])


current state: {'processed_until': '2026-09-15T12:47:19.077814+00:00'}
state file: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag/data/knowledge/helpgeorgia.state.json


## 11. Indexing (embeddings → Chroma)

Reads `data/knowledge/<chat>.fixed.jsonl` for every configured chat and **diffs** against what's already in the collection by a content hash (question+answer+type+city) — embeds/upserts only NEW or CHANGED units, deletes vectors whose unit is gone, leaves everything else untouched. See `src/index.py` docstring.

⚠️ Spends OpenAI tokens — but only for what actually changed.


In [106]:
# Or as a script:  uv run python -m src.index
from src.index import main as build_index

build_index()


[index] skip ipgeorgiachat: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag/data/knowledge/ipgeorgiachat.fixed.jsonl not found (run src.fix_knowledge first)
[index] done. +/~0 upserted, 0 deleted, 106 unchanged. Records in collection: 106


In [107]:
from src.store import get_collection
col = get_collection()
print("Records in collection:", col.count())

Records in collection: 106


## 12. Retrieval

top-k по косинусному сходству (`k`), затем отсекается всё ниже `config.RETRIEVAL_MIN_SCORE` (сейчас 0.5) — подобрано по реальным скорам: настоящее совпадение даёт 0.7+, а у запроса без ответа в базе скоры зажаты в 0.38-0.50 без явного лидера. `city`/`type`/`date` тут по-прежнему не фильтруют — это данные для LLM в `src/rag.py`, не для поиска.


In [122]:
from src.retrieve import search

MULTI_SOURCE_QUERIES = [
    "нужна ли страховка при въезде в Грузию?",  # 5 тредов
    "как перевести деньги из Грузии в Россию?",  # 3 треда
    "куда сдать старые батарейки?",  # 2 треда
]
NO_ANSWER_QUERIES = [  # recall показал, что это было в тредах, но не извлеклось — честное "не нашла" тут правильно
    "какая сейчас дорога Ахалцихе-Батуми?",
    "кто записывался в секцию интересов, куда пропала кнопка листа ожидания?",
    "где купить гранатовое домашнее вино?",
]
ONE_ANSWER_QUERIES = [  # recall показал, что это было в тредах, и извлеклось — один ответ
    "сколько стоит обмотать чемодан плёнкой в аэропорту",
    "как в грузии сделать справку об отсутствии судимости?",  # 1 тред
]

query = MULTI_SOURCE_QUERIES[2]

raw = search(query, k=8, min_score=0.1)      # всё, без фильтра
filtered = search(query, k=8)              # с фильтром (как в rag.py)
print(f"raw: {len(raw)}  ->  filtered (>= {__import__('config').RETRIEVAL_MIN_SCORE}): {len(filtered)}\n")
filtered_links = {h["meta"]["link"] for h in filtered}

for i, h in enumerate(raw, 1):
    m = h["meta"]
    cut = "" if m["link"] in filtered_links else "  <- отсечено порогом"
    print(f"--- #{i}  score={h['score']:.3f}  [{m['type']}] city={m.get('city') or None}{cut}")
    print(f"  Q: {m['question']}")
    print(f"  A: {m['answer'][:150]}")
    print()


raw: 8  ->  filtered (>= 0.5): 2

--- #1  score=0.764  [vote_based] city=None
  Q: Куда можно сдать старые батарейки и зарядные блоки на утилизацию?
  A: Боксы для утилизации есть в магазине Elite Electronics на вокзале; также стоит поискать такие боксы в торговых центрах. Рекомендации по пунктам можно 

--- #2  score=0.631  [vote_based] city=Тбилиси
  Q: Где в Тбилиси можно сдать использованные батарейки?
  A: Использованные батарейки принимают в East Point в главном здании на первом этаже. Также их можно сдать в кафе «Интроверт» во время свопов: там стоят б

--- #3  score=0.385  [vote_based] city=Тбилиси  <- отсечено порогом
  Q: Где в Тбилиси можно оставить упакованные велосипеды на хранение на две недели?
  A: Один из участников предлагает оставить велосипеды у него на балконе, в гараже или подвале.

--- #4  score=0.313  [vote_based] city=None  <- отсечено порогом
  Q: Как заказать зарядку для ноутбука Asus TUF 17 в Грузию, если она осталась в Польше?
  A: Можно заказать подходящую

## 13. RAG answer

Если фрагменты не отвечают на вопрос — модель добавляет краткий ответ из своих общих знаний (без интернета), явно помечая это как не из чата ("В чатах такого не обсуждали, но в целом известно..."). Ссылки теперь идут ПРЯМО В ТЕКСТЕ, сразу после факта: модель ставит номер фрагмента в скобках (`[2]`, при нескольких — `[2][4]`), а код на лету заменяет каждый такой номер на реальную ссылку + дату фрагмента (дата вычисляется из данных, а не пишется моделью — не может быть перепутана или забыта). Приватные источники (закрытые чаты) помечаются 🔒 перед ссылкой. Отдельного блока "ИСПОЛЬЗОВАНО: 1, 3" больше нет — это старый механизм. `sources` в возвращаемом dict остаётся (для отладки/аналитики), но в тексте ответа не дублируется (см. `src/rag.py::_inline_citations`). `MULTI_SOURCE_QUERIES` должны дать содержательный ответ с несколькими источниками прямо по тексту; `NO_ANSWER_QUERIES` — честный ответ из общих знаний модели с явной пометкой, а не пустое "не нашла".


In [110]:
from src.rag import answer

for q in MULTI_SOURCE_QUERIES + NO_ANSWER_QUERIES + ONE_ANSWER_QUERIES:
    res = answer(q)
    print("=" * 70)
    print("Q:", q)
    print(res["answer"])
    print("sources:", [s["link"] for s in res["sources"]])
    print()


Q: нужна ли страховка при въезде в Грузию?
Да, для временно въезжающих в Грузию туристов и бизнес-путешественников медицинская страховка указана как обязательная. Полис должен покрывать экстренную медицинскую помощь и другие предусмотренные расходы; в чатах упоминалось покрытие от 100 000 GEL.

На практике страховку при въезде часто не проверяют — в том числе при перелёте и через Ларс, — но это не отменяет формального требования. Поэтому лучше оформить полис заранее и иметь электронную или бумажную копию. Участники также сообщали, что полис Unison принимали; он был на английском и грузинском языках.

Для обладателей ВНЖ, по сообщениям из Батуми, страховка обычно не требовалась, хотя могли сделать устное предупреждение. Это опыт участников чата, а не официальное разъяснение, поэтому требования и практика проверки могут меняться.
sources: ['https://t.me/helpgeorgia/313784', 'https://t.me/helpgeorgia/313253', 'https://t.me/helpgeorgia/313036', 'https://t.me/helpgeorgia/313383', 'https://t

## 14. Debug: which prompt is actually sent to GPT

Useful to understand why the model answered the way it did, and to tweak the system prompt in `src/rag.py`.


In [123]:
from src.rag import _build_context, SYSTEM_PROMPT
from src.retrieve import search

q = "нужна ли страховка при въезде в Грузию?"
hits = search(q, k=8)

print("### SYSTEM PROMPT ###\n")
print(SYSTEM_PROMPT)
print("\n### CONTEXT (fragments) ###\n")
print(_build_context(hits))


### SYSTEM PROMPT ###

Ты — ассистент по жизни в Грузии. Отвечай на русском, опираясь В ПЕРВУЮ ОЧЕРЕДЬ на приведённые ниже фрагменты знаний, извлечённые и проверенные из Telegram-чатов. Это мнения и опыт людей из чатов, а не официальные источники — при необходимости делай оговорку.

Если фрагменты НЕ отвечают на вопрос (совсем или частично) — по оставшейся части коротко ответь из своих общих знаний, БЕЗ выдумывания фактов, которых не знаешь. Такой ответ явно отдели фразой вроде «В чатах такого не обсуждали, но в целом известно, что...» — не выдавай общие знания за опыт чата. Если и общих знаний нет — честно скажи, что не нашла ответа.

У каждого фрагмента указан тип:
- date_based — со временем меняется (цены, официальные требования). Если несколько фрагментов дают разные значения — доверяй более свежим по дате, но упомяни расхождение и не скрывай, что цифра могла устареть.
- vote_based — рекомендация/мнение/способ. Если фрагменты называют РАЗНЫЕ варианты (места, контакты, способы) — пе

## 15. Retrieval evaluation (порог × top_k, сетка)

Оценивает src/retrieve.py::search на золотом наборе вопросов (eval/golden_queries.jsonl, все 4+ чата + категория `critical` — темы, где нельзя ошибаться). Схема в два прохода, поэтому сам перебор порогов И top_k ничего не стоит: (1) для каждого вопроса один раз достаём топ-20 кандидатов БЕЗ порога и один раз просим LLM оценить релевантность каждого — (2) `sweep_grid` просто берёт первые k из уже отсортированного по score списка и фильтрует по порогу — чистый Python, без единого нового запроса к API.

**Что значит каждая колонка:**
- **precision** — из того, что реально дошло до генерации (прошло порог), какая доля действительно релевантна. Низкий precision = модели показывают мусор вперемешку с полезным.
- **recall** — из всех релевантных кандидатов, что вообще нашлись (в топ-20 без порога), какую долю не потеряли после отсечения. Низкий recall = порог/k слишком жёсткие.
- **f1** — гармоническое среднее precision и recall, одно число для сравнения комбинаций между собой.
- **hit_rate** — по скольким вопросам (кроме no_answer) осталась хотя бы одна релевантная запись. Грубее recall — просто "нашли хоть что-то или нет".
- **avg_hits** — сколько записей в среднем реально доходит до генерации. Влияет на стоимость и на риск "простыни" из шума в контексте.
- **no_answer_leak** — по вопросам, где ответа в базе НЕТ, какая доля всё равно "находит" что-то якобы релевantное. Должно быть ~0 — иначе система рискует уверенно наврать там, где данных просто нет.
- **critical_hit_rate** — по вопросам "нельзя ошибаться" (кроме scope_leakage-ловушек — там 0 совпадений и есть правильный результат), какая доля сохраняет хотя бы 1 релевантный источник. Если тут не 1.0 — минимум одному важному вопросу ретрив не даёт ни одного шанса на хороший ответ, независимо от промпта генерации.

**Как выбирать (порядок приоритета, см. `_pick_best`):** 1) `critical_hit_rate` ДОЛЖЕН быть 1.0 — не обсуждается; 2) `no_answer_leak` ДОЛЖЕН быть 0 — не обсуждается; 3) среди оставшихся комбинаций — максимальный `f1`; 4) при равенстве — меньше `avg_hits` (дешевле и меньше шума). Совет внизу таблицы уже применяет это правило, но финальное решение — твоё, после того как посмотришь и таблицу, и сами провалившиеся строки.

⚠️ Тратит токены: один эмбеддинг + один вызов config.JUDGE_MODEL на вопрос (~2 × число вопросов в golden-сете). Сам перебор (k, порог) — бесплатно.


In [ ]:
from src.eval_common import load_golden
from src.eval_retrieval import collect_run, sweep_grid, _report_grid

golden = load_golden()
print(f"{len(golden)} вопросов в golden-сете")

retrieval_rows = collect_run(golden)
_report_grid(sweep_grid(retrieval_rows))


## 16. RAG-answer evaluation (faithfulness / relevance / scope / citations)

Оценивает src/rag.py::answer на том же golden-сете, на ПРОДАКШН-конфиге (config.TOP_K, config.RETRIEVAL_MIN_SCORE) — то есть именно то поведение, которое реально уйдёт пользователю. Судья (config.JUDGE_MODEL, независимый от config.GENERATION_MODEL) проверяет 3 критерия через LLM: faithful (нет выдуманных фактов), relevant (отвечает по существу), scope_ok (не выдаёт грузинские факты за ответ про другую страну — см. историю с Испанией). Для вопросов категории `critical` добавляется honest_uncertainty (если источники противоречат — признал ли ответ это честно). Плюс две ДЕТЕРМИНИРОВАННЫЕ проверки без LLM: citation_leftover (не остался ли необработанный `[N]` в ответе) и date_suspect (не написала ли модель дату сама в прозе, а не через автоматическую подстановку).

Блок `CRITICAL` внизу отчёта печатается ПОЛНОСТЬЮ построчно всегда — один провал там не должен потеряться в общем проценте.

⚠️ Тратит токены: один вызов config.GENERATION_MODEL (сама система) + один вызов config.JUDGE_MODEL на вопрос.


In [ ]:
from src.eval_rag import run_eval, _report_rag

rag_rows = run_eval(golden)
_report_rag(rag_rows)
